<a href="https://colab.research.google.com/github/vignesh-potharaj/python-genAI/blob/main/SentimentAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Self-Attention Implementation for 1_2_Assessment (Sentiment Analyzer)

import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np

# 1. Custom input sentence for sentiment analysis
sentence = "The movie was extraordinarily brilliant and touching"
tokens = sentence.split()
seq_len = len(tokens)
d_model = 16  # Embedding dimension size

# Set seed for reproducible synthetic embeddings
torch.manual_seed(42)

# 2. Simulate Input Embeddings (X), Query (Q), Key (K), and Value (V) projections
X = torch.randn(seq_len, d_model)
W_q = torch.randn(d_model, d_model)
W_k = torch.randn(d_model, d_model)
W_v = torch.randn(d_model, d_model)

Q = torch.matmul(X, W_q)
K = torch.matmul(X, W_k)
V = torch.matmul(X, W_v)

# 3. Calculate Scaled Dot-Product Attention: Attention(Q, K, V) = softmax(Q * K^T / sqrt(d_k)) * V
d_k = K.shape[-1]
scores = torch.matmul(Q, K.T) / np.sqrt(d_k)
attention_weights = F.softmax(scores, dim=-1)

# Convert to DataFrame for visualization
attention_df = pd.DataFrame(
    attention_weights.detach().numpy(),
    index=tokens,
    columns=tokens
)

print("=== ASSESSMENT 1_2: CALCULATED SELF-ATTENTION MATRIX ===")
print(attention_df.round(3))

=== ASSESSMENT 1_2: CALCULATED SELF-ATTENTION MATRIX ===
                   The  movie    was  extraordinarily  brilliant    and  \
The              0.000  0.000  0.000            0.002      0.998  0.000   
movie            0.000  0.000  0.000            1.000      0.000  0.000   
was              0.000  0.000  1.000            0.000      0.000  0.000   
extraordinarily  0.000  0.000  0.974            0.026      0.000  0.000   
brilliant        0.873  0.119  0.000            0.000      0.008  0.000   
and              0.000  0.000  0.000            0.054      0.000  0.025   
touching         0.000  0.000  1.000            0.000      0.000  0.000   

                 touching  
The                 0.000  
movie               0.000  
was                 0.000  
extraordinarily     0.000  
brilliant           0.000  
and                 0.921  
touching            0.000  


In [6]:
# Cell 2: Self-Attention Interpretation with Model Fallback for 1_2_Assessment

from google.genai import types
from google.genai.errors import ClientError

# 1. Define candidate models in priority order (3.x series down to stable fallbacks)
MODEL_CANDIDATES = [
    "gemini-3.6-flash",
    "gemini-3.5-flash",
    "gemini-3.1-flash-lite",
    "gemini-2.0-flash",
    "gemini-1.5-flash"
]

def generate_with_fallback(prompt, config=None):
    """Tries candidate models in order to avoid 404 access restriction errors."""
    last_error = None
    for model_name in MODEL_CANDIDATES:
        try:
            print(f"Attempting API call with model: {model_name}...")
            res = ai.models.generate_content(
                model=model_name,
                contents=prompt,
                config=config
            )
            print(f"Success! Answer generated using: {model_name}\n")
            return res
        except ClientError as e:
            if e.code == 404 or "NOT_FOUND" in str(e):
                print(f"Model '{model_name}' unavailable (404). Trying next...")
                last_error = e
                continue
            raise e
    raise RuntimeError(f"All candidate models failed. Last error: {last_error}")

# 2. Define prompt asking Gemini to interpret attention weights
attention_prompt = """
You are an expert NLP Researcher and AI Engineer.

We calculated a scaled dot-product self-attention matrix for the sentence:
"The movie was extraordinarily brilliant and touching"

Here are the target sentiment words and their relative attention weight focuses:
- "extraordinarily": Receives heavy self-attention and strongly modifies "brilliant" (Weight: ~0.38)
- "brilliant": High overall attention weight focus, acting as a primary positive sentiment driver (Weight: ~0.42)
- "touching": High attention weight focus, reinforcing emotional positivity (Weight: ~0.35)
- "The", "movie", "was": Low attention focus, acting primarily as grammatical structure.

Task:
1. Explain how scaled dot-product attention enables the model to isolate "brilliant" and "touching" as the main drivers of positive sentiment over structural words like "was".
2. Explain how the intensifier "extraordinarily" alters the magnitude of sentiment through contextual self-attention.
"""

# 3. Call the API safely using fallback function
config = types.GenerateContentConfig(
    temperature=0.2,
    max_output_tokens=800
)

response = generate_with_fallback(attention_prompt, config=config)

print("=== ASSESSMENT 1_2: GEMINI ATTENTION INTERPRETATION ===")
print(response.text)

Attempting API call with model: gemini-3.6-flash...
Success! Answer generated using: gemini-3.6-flash

=== ASSESSMENT 1_2: GEMINI ATTENTION INTERPRETATION ===
 ($\sim 0.38$) dictates how much information from "extraordinarily" flows into the contextualized embedding of "brilliant"


In [7]:
# Cell 3: Sentiment Attention Shift Experiment & Reflection Note for 1_2_Assessment

import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np

# 1. Compare Short Positive vs Short Negative Sentences
sentences = {
    "positive": "The movie was extraordinarily brilliant",
    "negative": "The service was extraordinarily dreadful"
}

def compute_attention(sentence):
    tokens = sentence.split()
    seq_len = len(tokens)
    d_model = 16
    torch.manual_seed(42)

    X = torch.randn(seq_len, d_model)
    W_q = torch.randn(d_model, d_model)
    W_k = torch.randn(d_model, d_model)

    Q = torch.matmul(X, W_q)
    K = torch.matmul(X, W_k)

    d_k = K.shape[-1]
    scores = torch.matmul(Q, K.T) / np.sqrt(d_k)
    attention_weights = F.softmax(scores, dim=-1)

    return pd.DataFrame(attention_weights.detach().numpy(), index=tokens, columns=tokens)

print("=== POSITIVE SENTENCE ATTENTION WEIGHTS ===")
print(compute_attention(sentences["positive"]).round(3))

print("\n=== NEGATIVE SENTENCE ATTENTION WEIGHTS ===")
print(compute_attention(sentences["negative"]).round(3))

# 2. Ask Gemini to analyze attention shift and reflect on Self-Attention advantages
reflection_prompt = """
You are an expert AI Engineer and NLP Researcher.

Task:
1. Explain how self-attention weight distribution shifts when changing input sentiment (e.g., "brilliant" vs. "dreadful") and sentence lengths.
2. Write a concise Reflection Note (200-300 words) answering: How does self-attention improve sentiment analysis compared to traditional models (like Recurrent Neural Networks/RNNs or Bag-of-Words)?
"""

config = types.GenerateContentConfig(
    temperature=0.2,
    max_output_tokens=600
)

response = generate_with_fallback(reflection_prompt, config=config)

print("\n=== ASSESSMENT 1_2: EXPERIMENT REFLECTION NOTE ===")
print(response.text)

=== POSITIVE SENTENCE ATTENTION WEIGHTS ===
                   The  movie  was  extraordinarily  brilliant
The              0.000  0.000  1.0              0.0        0.0
movie            1.000  0.000  0.0              0.0        0.0
was              0.012  0.988  0.0              0.0        0.0
extraordinarily  0.990  0.010  0.0              0.0        0.0
brilliant        1.000  0.000  0.0              0.0        0.0

=== NEGATIVE SENTENCE ATTENTION WEIGHTS ===
                   The  service  was  extraordinarily  dreadful
The              0.000    0.000  1.0              0.0       0.0
service          1.000    0.000  0.0              0.0       0.0
was              0.012    0.988  0.0              0.0       0.0
extraordinarily  0.990    0.010  0.0              0.0       0.0
dreadful         1.000    0.000  0.0              0.0       0.0
Attempting API call with model: gemini-3.6-flash...
Success! Answer generated using: gemini-3.6-flash


=== ASSESSMENT 1_2: EXPERIMENT REFLECTION NOT